In [18]:
import os
import re
import dill 
import pandas as pd
import sys

sys.path.append(os.path.dirname(os.path.dirname(os.path.abspath(__name__))))


dir = "../data/Test_Pinning/"
csv_files = [file for file in os.listdir(dir) if file.endswith('.csv')]
dill_files = [file for file in os.listdir(dir) if file.endswith('.dill')]

csv_sub  = [f for f in os.listdir(dir) if f.startswith("Data_Graph_") and f.endswith(".csv")]
dill_sub = [f for f in os.listdir(dir) if f.startswith("JOB_Initial_Test_Nodes_20_Graph_") and f.endswith("_edges.dill")]
graphml_sub = [file for file in os.listdir(dir) if file.startswith('Graph_object') and file.endswith('.graphml')]

csv_by_id  = {int(re.search(r'\d+', f).group()) : f  for f in csv_sub}
dill_by_id = {int(re.findall(r'\d+', f)[1]): f for f in dill_sub} 
graphml_by_id = {int(re.search(r'\d+', f).group()): f for f in graphml_sub}


common_ids = csv_by_id.keys() & dill_by_id.keys()
if not common_ids:
    raise RuntimeError("No id appears in both CSV and dill files.")

# dill_by_id
# graphml_sub
# graphml_by_id
# # --- build the dictionary --------------------------------------------------
# pairs = {}
# for i in common_ids:
#     df_path   = os.path.join(dir, csv_by_id[i])
#     dill_path = os.path.join(dir, dill_by_id[i])

#     df = pd.read_csv(df_path)
#     with open(dill_path, "rb") as f:
#         graph = dill.load(f)

#     pairs[i] = (df, graph)



In [19]:
import networkx as nx
graphml_by_id[0]
G   = nx.read_graphml(f"{dir}/{graphml_by_id[0]}")

In [20]:
import networkx as nx
import pandas as pd
from pgmpy.models import DiscreteBayesianNetwork
from pgmpy.estimators import (
    BIC, K2, BDeu,
    MaximumLikelihoodEstimator, BayesianEstimator
)

def score_bn_from_nx(
    graph: nx.DiGraph,
    data: pd.DataFrame,
    *,
    estimator: str = "ml",   # "ml" or "bayesian"
    score: str = "bic",      # "bic", "k2", "bdeu"
    prior_type: str = "BDeu", # only used when estimator == "bayesian"
    equivalent_sample_size: int = 10,
) -> float:
    """
    Build a pgmpy BayesianModel from a networkx.DiGraph and return the requested
    structure score on `data`.

    Parameters
    ----------
    graph : nx.DiGraph
        The directed acyclic graph that encodes the Bayesian-network structure.
    data : pd.DataFrame
        A dataframe whose columns match the graph’s nodes and contain discrete values.
    estimator : {"ml", "bayesian"}, optional
        CPD-learning method. "ml" = maximum-likelihood; "bayesian" = BayesianEstimator.
    score : {"bic", "k2", "bdeu"}, optional
        Structure-scoring function to use.
    prior_type, equivalent_sample_size
        Passed straight to pgmpy’s BayesianEstimator if `estimator == "bayesian"`.

    Returns
    -------
    float
        The requested score (higher is better in pgmpy).
    """
    # 1) Build the model from the edge list
    model = DiscreteBayesianNetwork(graph.edges())
    
    # 2) Fit CPDs
    if estimator.lower() == "ml":
        model.fit(data, estimator=MaximumLikelihoodEstimator)
    elif estimator.lower() == "bayesian":
        model.fit(
            data,
            estimator=BayesianEstimator,
            prior_type=prior_type,
            equivalent_sample_size=equivalent_sample_size,
        )
    else:
        raise ValueError(f"Unknown estimator: {estimator}")

    # 3) Pick the scoring object
    score_obj = (
        BIC(data)  if score.lower() == "bic"  else
        K2(data)   if score.lower() == "k2"   else
        BDeu(data) if score.lower() == "bdeu" else
        None
    )
    if score_obj is None:
        raise ValueError(f"Unknown score: {score}")

    return score_obj.score(model)


In [21]:
df = pd.read_csv(f"{dir}/{csv_by_id[0]}")
real = score_bn_from_nx(graph=G, data=df)


INFO:pgmpy: Datatype (N=numerical, C=Categorical Unordered, O=Categorical Ordered) inferred from data: 
 {'0': 'N', '1': 'N', '2': 'N', '3': 'N', '4': 'N', '5': 'N', '6': 'N', '7': 'N', '8': 'N', '9': 'N', '10': 'N', '11': 'N', '12': 'N', '13': 'N', '14': 'N', '15': 'N', '16': 'N', '17': 'N', '18': 'N', '19': 'N'}


INFO:pgmpy: Datatype (N=numerical, C=Categorical Unordered, O=Categorical Ordered) inferred from data: 
 {'0': 'N', '1': 'N', '2': 'N', '3': 'N', '4': 'N', '5': 'N', '6': 'N', '7': 'N', '8': 'N', '9': 'N', '10': 'N', '11': 'N', '12': 'N', '13': 'N', '14': 'N', '15': 'N', '16': 'N', '17': 'N', '18': 'N', '19': 'N'}


In [22]:
from algorithms.benchmark import run_hc

results = run_hc(df)

INFO:pgmpy: Datatype (N=numerical, C=Categorical Unordered, O=Categorical Ordered) inferred from data: 
 {'0': 'N', '1': 'N', '2': 'N', '3': 'N', '4': 'N', '5': 'N', '6': 'N', '7': 'N', '8': 'N', '9': 'N', '10': 'N', '11': 'N', '12': 'N', '13': 'N', '14': 'N', '15': 'N', '16': 'N', '17': 'N', '18': 'N', '19': 'N'}
INFO:pgmpy: Datatype (N=numerical, C=Categorical Unordered, O=Categorical Ordered) inferred from data: 
 {'0': 'N', '1': 'N', '2': 'N', '3': 'N', '4': 'N', '5': 'N', '6': 'N', '7': 'N', '8': 'N', '9': 'N', '10': 'N', '11': 'N', '12': 'N', '13': 'N', '14': 'N', '15': 'N', '16': 'N', '17': 'N', '18': 'N', '19': 'N'}
INFO:pgmpy: Datatype (N=numerical, C=Categorical Unordered, O=Categorical Ordered) inferred from data: 
 {'0': 'N', '1': 'N', '2': 'N', '3': 'N', '4': 'N', '5': 'N', '6': 'N', '7': 'N', '8': 'N', '9': 'N', '10': 'N', '11': 'N', '12': 'N', '13': 'N', '14': 'N', '15': 'N', '16': 'N', '17': 'N', '18': 'N', '19': 'N'}


In [23]:
learned = score_bn_from_nx(results, df)

INFO:pgmpy: Datatype (N=numerical, C=Categorical Unordered, O=Categorical Ordered) inferred from data: 
 {'0': 'N', '1': 'N', '2': 'N', '3': 'N', '4': 'N', '5': 'N', '6': 'N', '7': 'N', '8': 'N', '9': 'N', '10': 'N', '11': 'N', '12': 'N', '13': 'N', '14': 'N', '15': 'N', '16': 'N', '17': 'N', '18': 'N', '19': 'N'}
INFO:pgmpy: Datatype (N=numerical, C=Categorical Unordered, O=Categorical Ordered) inferred from data: 
 {'0': 'N', '1': 'N', '2': 'N', '3': 'N', '4': 'N', '5': 'N', '6': 'N', '7': 'N', '8': 'N', '9': 'N', '10': 'N', '11': 'N', '12': 'N', '13': 'N', '14': 'N', '15': 'N', '16': 'N', '17': 'N', '18': 'N', '19': 'N'}


In [24]:
real > learned
print(real)
print(learned)

-64797.69550389212
-66416.87665176457
